# Softmax Without Ever Storing the Row

**Topic:** Online softmax &nbsp;·&nbsp; **Difficulty:** Hard

FlashAttention never materializes the `n x n` score matrix. The trick that makes
this possible is the online (streaming) softmax: process the row in blocks,
keeping a running maximum and a running sum, and rescale what you have whenever
the maximum moves.

Implement it and prove it is *exact*, not an approximation, by comparing
against the standard two-pass softmax to machine precision.

## The data (created in the next cell)

- `scores` `(4096,)`: one attention row, with one extreme entry so the naive
  one-pass version overflows.
- `V` `(4096, 64)`: the values to weight.
- `block = 256`.

## Your task

With **NumPy only**:

1. `reference`: the standard softmax (subtract the global max, exponentiate,
   normalize) times `V`.
2. `online`: process the row in blocks of 256, maintaining `m` (running max),
   `l` (running sum of exponentials), and `acc` (running weighted sum of `V`).
   On each block with max `m_b`: set `m_new = max(m, m_b)`, rescale
   `l <- l * exp(m - m_new)` and `acc <- acc * exp(m - m_new)`, then add the
   block's contribution, and finally divide `acc` by `l`.
3. Set `answers` to a dict with `"max_abs_diff"` (between the two outputs),
   `"final_l"`, `"final_m"`, and `"naive_overflowed"` (set to `1.0` if computing
   `exp(scores)` without any max subtraction produces a non-finite value,
   and `0.0` otherwise).

## Evaluation

Checked against fixed expected summary values (`final_l = 1.0`,
`final_m = 900.0`, `naive_overflowed = 1.0`) with tolerance
`1e-6 * (1 + abs(expected))` for each scalar and its fixed projection.
The evaluation cell also requires `max_abs_diff` to be below `1e-12`, and
checks that your `naive_overflowed` flag is `1.0`.

In [ ]:
import numpy as np

rng = np.random.default_rng(4008)
n, dv = 4096, 64
scores = rng.normal(0, 60, n)         # spread before injecting the extreme entry
scores[1234] = 900.0
V = rng.normal(0, 1, (n, dv))
block = 256
print(f"scores {scores.shape}, range {scores.min():.1f} to {scores.max():.1f}")

In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the data created above to compute the online softmax, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.

In [ ]:
# =====================================================================
# EVALUATION: run this cell to score your answer
# =====================================================================
import numpy as np

if "answers" not in dir():
    raise NameError("Define `answers` in the solution cell before running this cell.")
if not isinstance(answers, dict):
    raise TypeError("`answers` must be a dict.")

def _digest(a):
    a = np.asarray(a, dtype=float).ravel()
    if a.size == 0:
        return 0.0
    probe = np.random.default_rng(9176).standard_normal(a.size)
    return float(a @ probe)

_CHECKS = [
    ("max_abs_diff", [], 0.0, 0.0, 1e-06),
    ("final_l", [], 1.0, -0.16743801481449916, 1e-06),
    ("final_m", [], 900.0, -150.69421333304925, 1e-06),
    ("naive_overflowed", [], 1.0, -0.16743801481449916, 1e-06),
]

print("=" * 56)
_ok = True
for _key, _shape, _sum, _probe, _tol in _CHECKS:
    if _key not in answers:
        print(f"  {_key:<24} MISSING")
        _ok = False
        continue
    _a = np.asarray(answers[_key], dtype=float)
    if list(_a.shape) != _shape:
        print(f"  {_key:<24} shape {list(_a.shape)}, expected {_shape}")
        _ok = False
        continue
    if not np.all(np.isfinite(_a)):
        print(f"  {_key:<24} contains NaN or inf")
        _ok = False
        continue
    _g = (abs(_a.sum() - _sum) <= _tol * (1.0 + abs(_sum))
          and abs(_digest(_a) - _probe) <= _tol * (1.0 + abs(_probe)))
    print(f"  {_key:<24} {'ok' if _g else 'wrong'}")
    _ok = _ok and _g

if "max_abs_diff" in answers:
    _exact = answers["max_abs_diff"] < 1e-12
    _of = float(answers["naive_overflowed"]) == 1.0
    print(f"  {'online is exact':<24} {'ok' if _exact else 'wrong'}  "
          f"(max diff {answers['max_abs_diff']:.2e})")
    print(f"  {'naive exp overflows':<24} {'ok' if _of else 'wrong'}")
    _ok = _ok and _exact and _of

print("=" * 56)
if _ok:
    print("PASSED, nice work!")
else:
    print("NOT PASSED yet. Revisit your solution and try again.")

## Solution

The rescaling identity is the whole algorithm: if you have accumulated with
reference max `m` and meet a larger max `m_new`, every stored term is off by
exactly `exp(m - m_new)`, so one multiply fixes the entire accumulator. Because
that correction is exact, the streaming result equals the two-pass result to
floating-point precision. FlashAttention is not an approximation, which is why
it could be adopted without any accuracy discussion at all. The memory saving is
what falls out: you never hold more than one block of scores.

```python
import numpy as np

m_all = scores.max()
w = np.exp(scores - m_all)
reference = (w[:, None] * V).sum(axis=0) / w.sum()

m = -np.inf
l = 0.0
acc = np.zeros(dv)
for start in range(0, len(scores), block):
    sb = scores[start:start + block]
    Vb = V[start:start + block]
    mb = float(sb.max())
    m_new = max(m, mb)
    scale = np.exp(m - m_new) if np.isfinite(m) else 0.0
    p = np.exp(sb - m_new)
    l = l * scale + float(p.sum())
    acc = acc * scale + p @ Vb
    m = m_new
online = acc / l

with np.errstate(over="ignore"):
    naive = np.exp(scores)
overflowed = float(not np.all(np.isfinite(naive)))

answers = {
    "max_abs_diff": float(np.abs(reference - online).max()),
    "final_l": float(l),
    "final_m": float(m),
    "naive_overflowed": overflowed,
}
```